# Survival Simulator - baseline runner

Execution trigger only: all logic lives in `agents/` and `training/` (.py files).

* **Scored runs** (logged to `results/index.csv`) are launched as a **fresh subprocess** via `!python -m training.evaluate`, so kernel state can never leak into a result.
* **Rendering** runs in-kernel and is for watching the game only; its scores are never logged.

Run this notebook with its working directory = `survival-simulator/`.

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}


In [ ]:
import os
os.chdir("/home/jovyan/Nordic-AI-cup-2026")
!git checkout challenge-1
print(os.listdir("."))

In [ ]:
import os, sys, subprocess, glob


# Must run from survival-simulator/ so `src`, `agents`, `training` import.
# JupyterHub kernels may start in $HOME, so locate the folder and cd into it.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])
print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(sys.executable)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# Warn if this kernel's packages differ from the pinned simulator requirements
from importlib.metadata import version, PackageNotFoundError
for line in open("requirements.txt"):
    name, _, pinned = line.strip().partition("==")
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = "MISSING"
    if installed != pinned:
        print(f"WARNING: {name} installed={installed} pinned={pinned}")


## 0. One-time install (skip if already installed)

In [ ]:
#!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## 1. Tests (run before every scored run)

In [ ]:
!{sys.executable} -m pytest -q tests

## 2. Scored evaluation (fresh process)

Parameters to set per experiment: experiment id, config, seeds, workers (cluster: 4 CPUs), per-game wall-clock cap.
Commit before running - the run records the git revision and flags `-dirty` trees.

In [ ]:
EXPERIMENT = "C1-E00"
CONFIG = "training/configs/dummy_v0.json"
SEEDS = "0 1 2 3 4"
WORKERS = 4
MAX_WALL_SEC = 1800

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

In [ ]:
EXPERIMENT = "C1-E01"
CONFIG = "training/configs/heuristic_v0.json"

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

## 3. Results index

In [ ]:
import pandas as pd
pd.read_csv("results/index.csv")

## 4. Watch a game (inspection only, not scored)

`max_sim_time` limits how much of the game is rendered (one frame per simulated second by default).

In [ ]:
from IPython.display import Video, Image, display
from training.render import render_episode

video_path, stats = render_episode("training/configs/heuristic_v0.json", seed=0, max_sim_time=300, frame_every=10, width=640, fps=20)
display(Video(os.path.relpath(video_path), embed=False) if video_path.endswith(".mp4") else Image(filename=video_path))
stats